# RQ2 (Moodle): temporal engagement profiles from timestamped events
Revised analysis. Changes from the first version:
1. Exam-sitting events (all `IT_EE_*` end-exam quiz events) and system auto-saves are removed, and the study period is limited to 23 Oct 2025 – 8 Feb 2026 (end of the exam window).
2. Sessions per week uses the fixed 15.6-week study period as denominator.
3. Clustering features are log-transformed before standardising; k is chosen on silhouette, bootstrap stability and agreement with Ward clustering.
4. Profiles are validated against behaviour not used to build them; transition differences are tested with a permutation test and FDR correction.

In [ ]:
import pandas as pd, numpy as np
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, adjusted_rand_score
m=pd.read_csv('../data/moodle/logs.csv')
m['Time']=pd.to_datetime(m['Time'],format='%d/%m/%y, %H:%M:%S')
uid=m['User ID'].astype(str).str.strip("'").str.extract(r'^(\d+)$',expand=False)
m['uid']=uid.fillna(m['Description'].str.extract(r"user with (?:the )?id '(\d+)'",expand=False)).astype(int)
print('Events:',len(m),'| learners:',m.uid.nunique())

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
C=['#2a78d6','#eb6834','#1baf7a','#eda100']
INK='#0b0b0b'; INK2='#52514e'; GRID='#e4e3de'; SURF='#ffffff'
mpl.rcParams.update({'font.family':'DejaVu Sans','font.size':7,'axes.edgecolor':GRID,'axes.labelcolor':INK2,'xtick.color':INK2,'ytick.color':INK2,
 'axes.grid':True,'grid.color':GRID,'grid.linewidth':0.8,'axes.spines.top':False,'axes.spines.right':False,'axes.titlesize':7.5,'axes.titlecolor':INK,
 'axes.titleweight':'bold','axes.titlelocation':'left','lines.linewidth':1.5,'lines.solid_capstyle':'round','legend.frameon':False,'figure.dpi':200,'savefig.dpi':200,
 'savefig.bbox':'tight','figure.facecolor':SURF,'axes.facecolor':SURF,'xtick.major.size':0,'ytick.major.size':0,'legend.fontsize':6.8})

import os, pandas as pd; os.makedirs('figures',exist_ok=True); pd.set_option('display.width',220); pd.set_option('display.max_columns',30)

## 1. Filtering and sessionisation

In [ ]:
START=pd.Timestamp('2025-10-23'); END=pd.Timestamp('2026-02-09')
auto=m['Event name']=='Quiz attempt auto-saved'; exam=m['Event context'].str.startswith('Quiz: IT_EE'); late=m.Time>=END
print('Auto-saves:',auto.sum(),'| exam-quiz events:',exam.sum(),'| both:',(auto&exam).sum(),'| removed as exam/system:',(auto|exam).sum())
print('Removed as after 8 Feb (not already removed):',(late&~(auto|exam)).sum())
s=m[~auto&~exam&~late].sort_values(['uid','Time']).copy()
print('Study events kept:',len(s),'| learners:',s.uid.nunique())
gap=s.groupby('uid').Time.diff()
s['session']=(gap.isna()|(gap>pd.Timedelta(minutes=30))).groupby(s.uid).cumsum()
S=s.groupby(['uid','session']).agg(start=('Time','min'),end=('Time','max'),events=('Time','size')).reset_index()
S['duration']=(S.end-S.start).dt.total_seconds()/60; S['day']=S.start.dt.normalize()
print('Sessions:',len(S))

## 2. Learner features

In [ ]:
weeks=(END-START).days/7
L=S.groupby('uid').agg(sessions=('session','size'),active_days=('day','nunique'),median_session_duration=('duration','median'),median_events_per_session=('events','median'))
L['sessions_per_week']=L.sessions/weeks
d=S[['uid','day']].drop_duplicates().sort_values(['uid','day']); d['g']=d.groupby('uid').day.diff().dt.days
L['median_active_day_gap']=d.groupby('uid').g.median()
L=L[(L.sessions>=3)&(L.active_days>=3)].copy()
F=['sessions_per_week','median_session_duration','median_events_per_session','median_active_day_gap']
print('Learners kept:',len(L)); print(L[F].describe().round(2))

### Sensitivity to the session threshold

In [ ]:
gm=s.sort_values(['uid','Time']).groupby('uid').Time.diff().dt.total_seconds().div(60).dropna()
print((pd.cut(gm,[0,5,10,15,30,60,120,np.inf],right=False).value_counts(sort=False)/len(gm)*100).round(2))
def features_at(thr):
    d=s.sort_values(['uid','Time']).copy(); gg=d.groupby('uid').Time.diff()
    d['sn']=(gg.isna()|(gg>pd.Timedelta(minutes=thr))).groupby(d.uid).cumsum()
    SS=d.groupby(['uid','sn']).agg(st=('Time','min'),en=('Time','max'),n=('Time','size')).reset_index()
    SS['dur']=(SS.en-SS.st).dt.total_seconds()/60; SS['day']=SS.st.dt.normalize()
    Fx=SS.groupby('uid').agg(ns=('sn','size'),dur=('dur','median'),ev=('n','median')); Fx['spw']=Fx.ns/weeks
    dd=SS[['uid','day']].drop_duplicates().sort_values(['uid','day']); dd['g']=dd.groupby('uid').day.diff().dt.days; Fx['gap']=dd.groupby('uid').g.median()
    return Fx.reindex(L.index), len(SS)
SENS={}
for thr in [10,15,30,60,120]:
    Fx,ns=features_at(thr); Xs=StandardScaler().fit_transform(np.log1p(Fx[['spw','dur','ev','gap']].fillna(0)))
    SENS[thr]=KMeans(4,random_state=42,n_init=50).fit_predict(Xs); print(f'{thr} min: {ns} sessions')

## 3. Choosing k

In [ ]:
X=StandardScaler().fit_transform(np.log1p(L[F]))
rows=[]
for k in range(2,7):
    ref=KMeans(k,random_state=42,n_init=50).fit_predict(X)
    ward=AgglomerativeClustering(k,linkage='ward').fit_predict(X)
    rng=np.random.default_rng(0); boot=[]
    for b in range(100):
        idx=rng.choice(len(X),len(X),replace=True)
        boot.append(adjusted_rand_score(ref,KMeans(k,random_state=b,n_init=10).fit(X[idx]).predict(X)))
    rows.append([k,silhouette_score(X,ref),np.mean(boot),adjusted_rand_score(ref,ward),' / '.join(map(str,sorted(np.bincount(ref),reverse=True)))])
print(pd.DataFrame(rows,columns=['k','silhouette','bootstrap ARI','Ward ARI','sizes']).round(2).to_string(index=False))

In [ ]:
lab=KMeans(4,random_state=42,n_init=50).fit_predict(X)
L['raw']=lab
med=L.groupby('raw')[F].median()
# name clusters from their medians
names={}
names[med.sessions_per_week.idxmax()]='Frequent & regular'
names[med.median_active_day_gap.idxmax()]='Sporadic'
names[med.median_session_duration.drop(list(names)).idxmax()]='Long sessions'
names[[i for i in range(4) if i not in names][0]]='Brief check-ins'
L['cluster']=L.raw.map(names); order=['Frequent & regular','Brief check-ins','Long sessions','Sporadic']; col=dict(zip(order,C))
print(L.groupby('cluster')[F].median().loc[order].round(2).assign(n=L.cluster.value_counts()))

In [ ]:
for thr,labs_ in SENS.items():
    tab=pd.DataFrame({'c':labs_,'syn':L.index.isin(sub[sub['Event context']=='Assignment: Synopsis'].uid) if 'sub' in globals() else 0})
    print(f'{thr} min: ARI vs 30-min profiles {adjusted_rand_score(L.cluster,labs_):.2f}')

## 4. Validation on behaviour not used for clustering

In [ ]:
sub=m[m['Event name']=='A submission has been submitted.']
L['synopsis']=L.index.isin(sub[sub['Event context']=='Assignment: Synopsis'].uid); L['any_assignment']=L.index.isin(sub.uid)
W0=pd.Timestamp('2026-01-26')
ex=m[(m['Event name']=='Quiz attempt started')&m['Event context'].str.startswith('Quiz: IT_EE')&(m.Time>=W0)&(m.Time<END)]
first=ex.groupby('uid').Time.min(); L['exam_attempted']=L.index.isin(first.index); L['exam_start_day']=((first-W0).dt.total_seconds()/86400).reindex(L.index)
pq=m[(m['Event name']=='Quiz attempt started')&m['Event context'].str.contains('IT_PQ|MKC')&(m.Time<END)]
L['practice_quiz_attempts']=pq.groupby('uid').size().reindex(L.index).fillna(0)
print(L.groupby('cluster').agg(n=('raw','size'),synopsis=('synopsis','mean'),any_assignment=('any_assignment','mean'),exam_attempted=('exam_attempted','mean'),
      practice_quiz_mean=('practice_quiz_attempts','mean'),exam_start_day_median=('exam_start_day','median')).loc[order].round(2))
for c in ['synopsis','any_assignment','exam_attempted']:
    chi,p,_,_=stats.chi2_contingency(pd.crosstab(L.cluster,L[c])); print(f'{c}: chi2={chi:.1f}, p={p:.2g}')
for c in ['practice_quiz_attempts','exam_start_day']:
    H,p=stats.kruskal(*[L.loc[L.cluster==k,c].dropna() for k in order]); print(f'{c}: Kruskal-Wallis H={H:.1f}, p={p:.2g}')

### Sensitivity of the validation to k (not used to choose k)

In [ ]:
for k in [3,4,5]:
    ck=KMeans(k,random_state=42,n_init=50).fit_predict(X)
    tab=pd.DataFrame({'c':ck,'synopsis':L.synopsis.values,'exam':L.exam_attempted.values,'spw':L.sessions_per_week.values,'gap':L.median_active_day_gap.values})
    print(f'k={k}'); print(tab.groupby('c').agg(n=('spw','size'),sessions_per_week=('spw','median'),gap=('gap','median'),synopsis=('synopsis','mean'),exam=('exam','mean')).sort_values('sessions_per_week',ascending=False).round(2))
    for v_ in ['synopsis','exam']:
        chi,p,_,_=stats.chi2_contingency(pd.crosstab(tab.c,tab[v_])); print(f'   {v_}: chi2={chi:.1f}, p={p:.2g}')

## 5. Figure 2 (cohort weekly, each feature as % of its term average), Figure 4 (raincloud plots); Table 5 medians

In [ ]:
LL=L.rename(columns={'sessions_per_week':'spw','median_session_duration':'dur','median_events_per_session':'ev','median_active_day_gap':'gap'})
from scipy.stats import gaussian_kde
# raincloud
FR=[('spw','(a) Sessions/week'),('dur','(b) Session min'),('ev','(c) Events/session'),('gap','(d) Gap (days)')]
fig,axs=plt.subplots(1,4,figsize=(5.8,2.5)); axs=axs.ravel(); rng=np.random.default_rng(0)
for ax,(fcol,t) in zip(axs,FR):
    allv=np.log10(LL[fcol].clip(lower=0.1)); lo,hi=allv.min()-0.1,allv.max()+0.1; xs=np.linspace(lo,hi,300)
    for j,c in enumerate(order):
        base=len(order)-1-j; v=np.log10(LL.loc[LL.cluster==c,fcol].clip(lower=0.1)).values
        if len(np.unique(v))>2:
            k=gaussian_kde(v,bw_method=0.45)(xs); k=k/k.max()*0.42
            ax.fill_between(xs,base+0.05,base+0.05+k,color=col[c],alpha=0.55,lw=0)
        q1,med,q3=np.percentile(v,[25,50,75])
        ax.plot([q1,q3],[base-0.05,base-0.05],color=INK,lw=3,solid_capstyle='butt'); ax.plot([med],[base-0.05],'o',color='white',ms=3.5,markeredgecolor=INK,markeredgewidth=0.8)
        ax.scatter(v,base-0.25+rng.uniform(-0.08,0.08,len(v)),s=5,color=col[c],alpha=0.7,linewidths=0)
    ax.set_yticks(range(4))
    if ax is axs[0]: ax.set_yticklabels(['Frequent &\nregular','Brief\ncheck-ins','Long\nsessions','Sporadic'][::-1],fontsize=8)
    else: ax.tick_params(labelleft=False); ax.grid(axis='y',visible=False)
    tv=[0.1,0.3,1,3,10,30,100]; tv=[x for x in tv if lo<=np.log10(x)<=hi]; ax.set_xticks(np.log10(tv)); ax.set_xticklabels([f'{x:g}' for x in tv],fontsize=7.5)
    ax.set_xlim(lo,hi); ax.set_ylim(-0.5,3.55); ax.set_title(t,fontsize=8.5)
plt.tight_layout(w_pad=0.4); plt.savefig('figures/'+'moodle_raincloud.png')
# ---- Figure 5: heatmap of profile medians
FH=[('spw','Sessions\nper week','{:.2f}'),('dur','Session\nlength (min)','{:.1f}'),('ev','Events per\nsession','{:.1f}'),('gap','Gap between\nactive days','{:.0f} d')]
lg=np.log1p(LL[[f for f,_,_ in FH]]); z=(lg-lg.mean())/lg.std(); z['cluster']=LL.cluster
Z=z.groupby('cluster').median().loc[order]; M=LL.groupby('cluster')[[f for f,_,_ in FH]].median().loc[order]
# Option A: heatmap
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
cmap=LinearSegmentedColormap.from_list('div',['#2a78d6','#f2f1ec','#eb6834'])
fig,ax=plt.subplots(figsize=(5.2,2.4))
im=ax.imshow(Z.values,cmap=cmap,norm=TwoSlopeNorm(0,-2,2),aspect='auto')
for i in range(4):
    for j,(f,_,fmt) in enumerate(FH):
        ax.text(j,i,fmt.format(M.iloc[i,j]),ha='center',va='center',fontsize=9.5,color=INK)
ax.set_xticks(range(4)); ax.set_xticklabels([l for _,l,_ in FH],fontsize=8.5); ax.set_yticks(range(4)); ax.set_yticklabels(order,fontsize=9)
ax.tick_params(length=0); ax.grid(False); [s.set_visible(False) for s in ax.spines.values()]; ax.xaxis.tick_top()
cb=fig.colorbar(im,ax=ax,fraction=0.04,pad=0.02); cb.set_label('Below ← cohort average → above',fontsize=7.5); cb.ax.tick_params(labelsize=7)
plt.tight_layout(); plt.savefig('figures/moodle_heatmap.png')
# ---- Figure 2: cohort weekly, one graph, each feature indexed to its own term average
d=s[s.uid.isin(L.index)].copy(); d['sn']=d['session']
S=d.groupby(['uid','sn']).agg(st=('Time','min'),en=('Time','max'),n=('Time','size')).reset_index()
S['dur']=(S.en-S.st).dt.total_seconds()/60; S['day']=S.st.dt.normalize(); S['week']=((S.day-START).dt.days//7)+1
Wk=int(S.week.max())
ad=S.groupby(['uid','week']).day.nunique().reindex(pd.MultiIndex.from_product([L.index,range(1,Wk+1)],names=['uid','week']),fill_value=0).reset_index(name='ad')
days=S[['uid','day','week']].drop_duplicates(['uid','day']).sort_values(['uid','day']); days['gap']=days.groupby('uid').day.diff().dt.days
ser={'Frequency (active days per week)':ad.groupby('week').ad.mean(),'Session length (median minutes)':S.groupby('week').dur.median(),
     'Events per session (median)':S.groupby('week').n.median(),'Gap between active days (median days)':days.dropna().groupby('week').gap.median()}
def wkof(t): return (pd.Timestamp(t)-START).days/7+1
fig,ax=plt.subplots(figsize=(5.8,2.7))
ax.axvspan(wkof('2026-01-26')-0.5,wkof('2026-02-09')-0.5,color='#efeeea',zorder=0); ax.text(wkof('2026-02-02')-0.5,203,'Exam\nwindow',ha='center',va='top',fontsize=8,color=INK2)
for t_,lab in [('2025-11-30','A'),('2026-01-04','B'),('2026-01-25','C')]:
    x=wkof(t_)-0.5; ax.axvline(x,color=INK2,lw=0.7,ls=(0,(2,2))); ax.text(x-0.1,212,lab,fontsize=8.5,color=INK,ha='right',va='top',fontweight='bold')
ax.axhline(100,color=INK2,lw=0.8)
for i,(k,v) in enumerate(ser.items()):
    idx=v/v.mean()*100; ax.plot(idx.index,idx.values,color=C[i],label=k,lw=1.7)
ticks=[1,3,5,7,9,11,13,15]; ax.set_xticks(ticks); ax.set_xticklabels([(START+pd.Timedelta(weeks=t-1)).strftime('%d %b') for t in ticks],fontsize=8)
ax.set_ylim(0,215); ax.set_xlim(0.5,Wk+0.5); ax.set_ylabel('% of the feature\'s term average'); ax.set_xlabel('Week starting')
ax.legend(loc='upper left',ncol=2,fontsize=7.8,bbox_to_anchor=(0,-0.2))
plt.tight_layout(); plt.savefig('figures/'+'moodle_cohort_weekly.png')
for k,v in ser.items(): print(k, 'mean',round(v.mean(),2),'max idx',round((v/v.mean()*100).max()),'at week',int((v/v.mean()).idxmax()))
print(LL.groupby('cluster')[['spw','dur','ev','gap']].median().loc[order].round(2).assign(n=LL.cluster.value_counts()))

## 6. Transition Network Analysis: transition probabilities and permutation test
Transition probabilities per profile and the learner-level permutation test (2,000 shuffles of profile labels, Benjamini–Hochberg correction) reported in Section 5.4. Figure 6 itself is drawn in Carm TNA from the file exported in section 8.

In [ ]:
state={'Quiz':'Quiz','SCORM package':'SCORM','Assignment':'Assignment','File submissions':'Assignment','Online text submissions':'Assignment','Book':'Content','Page':'Content','File':'Content','Glossary':'Content','Forum':'Discussion','System':'Course'}
St=['Course','Content','SCORM','Quiz','Assignment','Discussion']
q=s[s.uid.isin(L.index)].copy(); q['state']=q.Component.map(state); q['sid']=q.uid.astype(str)+'_'+q.session.astype(str)
q=q[q.state!=q.groupby('sid').state.shift()]; q['nxt']=q.groupby('sid').state.shift(-1)
T=q.dropna(subset=['nxt'])
cnt=T.groupby(['uid','state','nxt']).size().unstack(['state','nxt'],fill_value=0).reindex(columns=pd.MultiIndex.from_product([St,St]),fill_value=0)
labs=L.cluster.reindex(cnt.index).values; A=cnt.values
def probs(mask):
    t=A[mask].sum(axis=0).reshape(6,6); return t/np.maximum(t.sum(1,keepdims=True),1), t
rng=np.random.default_rng(1); res=[]
for k in order:
    def diff(lb):
        pa,ta=probs(lb==k); pb,tb=probs(lb!=k); return pa-pb,ta.sum(1),tb.sum(1)
    obs,na,nb_=diff(labs); null=np.array([diff(rng.permutation(labs))[0] for _ in range(2000)])
    p=(np.abs(null)>=np.abs(obs)).mean(0)
    for i in range(6):
        for j in range(6):
            if i!=j and na[i]>=30 and nb_[i]>=30: res.append((k,St[i],St[j],obs[i,j],p[i,j]))
R=pd.DataFrame(res,columns=['profile','from','to','difference','p'])
o=np.argsort(R.p.values); qv=np.empty(len(R)); prev=1
for r_,i in enumerate(o[::-1]): prev=min(prev,R.p.values[i]*len(R)/(len(R)-r_)); qv[i]=prev
R['q_BH']=qv
print('Transitions tested:',len(R),'| significant after FDR (q<0.05):',(R.q_BH<0.05).sum())
print(R.sort_values('p').head(8).round(3).to_string(index=False))

## 7. Supplementary supervised check: do pre-deadline timing features predict participation?
Synopsis submission and end-exam attempt are used as **targets**, never as clustering features. Features are recomputed from study events before each cutoff (one week before the Synopsis deadline; the day the exam window opens), excluding assignment-page events so that submitting does not feed its own predictor. Logistic Regression, 5-fold stratified CV repeated 20 times.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
s_noassign=s[~s.Component.isin(['Assignment','File submissions','Online text submissions'])]
def pre_features(end):
    d=s_noassign[(s_noassign.uid.isin(L.index))&(s_noassign.Time<end)].sort_values(['uid','Time']).copy()
    g_=d.groupby('uid').Time.diff(); d['sn']=(g_.isna()|(g_>pd.Timedelta(minutes=30))).groupby(d.uid).cumsum()
    SS=d.groupby(['uid','sn']).agg(st=('Time','min'),en=('Time','max'),n=('Time','size')).reset_index()
    SS['dur']=(SS.en-SS.st).dt.total_seconds()/60; SS['day']=SS.st.dt.normalize()
    X_=SS.groupby('uid').agg(ns=('sn','size'),dur=('dur','median'),ev=('n','median')).reindex(L.index)
    X_['spw']=X_.ns.fillna(0)/((end-START).days/7)
    dd=SS[['uid','day']].drop_duplicates().sort_values(['uid','day']); dd['g']=dd.groupby('uid').day.diff().dt.days
    X_['gap']=dd.groupby('uid').g.median().reindex(L.index).fillna((end-START).days)
    return np.log1p(X_[['spw','dur','ev','gap']].fillna(0))
rcv=RepeatedStratifiedKFold(n_splits=5,n_repeats=20,random_state=0)
for target,end in [('exam_attempted',pd.Timestamp('2026-01-26')),('synopsis',pd.Timestamp('2025-11-24'))]:
    Xp=pre_features(end); yv=L[target].astype(int).values
    print(f'{target}: cutoff {end.date()}, positives {yv.sum()} of {len(yv)}')
    for cols in [['spw'],['spw','dur','ev','gap']]:
        sc_=cross_val_score(make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000,class_weight='balanced')),Xp[cols],yv,cv=rcv,scoring='roc_auc')
        print(f'   {cols}: AUC {sc_.mean():.3f} ± {sc_.std():.3f}')

## 8. Export sessions for Carm TNA (Figure 6)
One row per collapsed event: `actor` = session id (so each session is one sequence and Carm does not merge or re-split sessions), `order` = position in the session, `action` = state, `group` = profile. Upload `tna_upload_sessions.csv` to the Carm TNA notebook, then Build > Groups > Build Group Networks.

In [ ]:
q2=q.copy(); q2['order']=q2.groupby('sid').cumcount()+1
carm=pd.DataFrame({'actor':q2.sid.values,'order':q2.order.values,'action':q2.state.values,'group':L.cluster.reindex(q2.uid).values})
carm.to_csv('tna_upload_sessions.csv',index=False); print(carm.shape,'saved tna_upload_sessions.csv (load into Carm TNA, then Build > Groups > Build Group Networks)')

In [ ]:
L.reset_index().rename(columns={'uid':'User ID'})[['User ID']+F+['cluster','synopsis','any_assignment','exam_attempted','practice_quiz_attempts']].to_csv('RQ2_Moodle_features.csv',index=False)
print('saved RQ2_Moodle_features.csv')